<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "296"
iso_country = "Kiribati"
year = 2020
import_portfolio = "h5_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 222.000*1e6

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Kiribati': 

Number of trade records present in dataset: 6733
Number of partner countries present in dataset: 44
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$101.7m
Total weight within the dataset: 112.9 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,296,Kiribati,156,China,Import,Sea,151530,151530 | Vegetable oils; cast,1515,1515 | Fixed vegetable or m,15,"15 | Animal, vegetable or",2167,"Palm, coconut, palm kernel, babassu and linsee...",Agriculture,21,VOL - Vegetable oils and fats,2732.118,0,2732.118,kg,7680.0,7680.0,1368.244694,8584.0,5333.848664,7298.010135,"['Container', 'Chemical tanker']","['Container', 'Chemical tanker']",38.145452,38.078849,38.085358,38.085358,38.115204,38.115204,39.902083,70.719010,44.013276,40.862231,40.973204,41.999413,39.071147,69.258441,46.326749,42.434610,41.460819,43.338582,38.342376,75.313860,57.447392,50.241462,46.482863,45.171638,37.627106,80.077089,65.894248,61.748503,49.765886,48.412692,36.925338,81.146913,77.528695,78.195293,53.951391,50.855972,36.209503,79.479024,76.165256,83.450229,61.727559,51.788084,35.535135,70.897218,73.893773,77.432963,58.046197,52.399986,34.847494,65.125154,66.201423,82.577731,68.220793,52.746983,2167
1,1,296,Kiribati,242,Fiji,Import,Sea,151530,151530 | Vegetable oils; cast,1515,1515 | Fixed vegetable or m,15,"15 | Animal, vegetable or",2167,"Palm, coconut, palm kernel, babassu and linsee...",Agriculture,21,VOL - Vegetable oils and fats,267.018,0,267.018,kg,5.0,5.0,133.722614,3362.0,2089.049302,279.353134,"['Container', 'Chemical tanker']","['Container', 'Chemical tanker']",1.460131,1.457582,1.457831,1.457831,1.458973,1.458973,1.527371,2.706981,1.684740,1.564124,1.568372,1.607653,1.495565,2.651074,1.773295,1.624311,1.587037,1.658914,1.467669,2.882863,2.198970,1.923142,1.779270,1.729079,1.440290,3.065190,2.522299,2.363608,1.904938,1.853140,1.413428,3.106140,2.967642,2.993158,2.065151,1.946664,1.386027,3.042297,2.915453,3.194307,2.362807,1.982344,1.360213,2.713803,2.828505,2.963978,2.221892,2.005766,1.333892,2.492860,2.534057,3.160909,2.611355,2.019048,2167


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $42.7m (42.0%), ... 

                            cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                          
10 | Cereals               7369632.1  12808175.8      7.2     11.3
12 | Oil seeds and oleagi  6369711.9   1759553.9      6.3      1.6
16 | Meat, fish, crustace  5034152.9   1104985.7      5.0      1.0
19 | Preparations of cere  4522482.5   2203318.7      4.4      2.0
17 | Sugars and sugar con  3766777.8   7667006.7      3.7      6.8
02 | Meat and edible meat  3739233.0   1942902.4      3.7      1.7
22 | Beverages, spirits a  2464968.3  17527766.9      2.4     15.5
04 | Dairy produce; birds  2150196.0    864876.1      2.1      0.8
11 | Products of the mill  1790297.0   4357366.5      1.8      3.9
21 | Miscellaneous edible  1600260.6    598472.1      1.6      0.5
07 | Vegetables and certa   718346.0   1052401.3      0.7      0.9
20 | Preparations of vege   7

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $38.2m (37.6%), ... 

                    cifvalue  vol_kg_vtw  pct_val  pct_vol
eora_desc                                                 
Agriculture       22863757.5  22964387.5     22.5     20.3
Food & Beverages  15370009.6  22170492.5     15.1     19.6
Fishing               9140.6      1846.6      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $35.5m (34.9%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     10743434.4   8304963.3     10.6      7.4
OMT - Meat products n.e.c.      7540346.0   2792543.9      7.4      2.5
PCR - Processed rice            7366165.7  12795029.8      7.2     11.3
OCR - Crops n.e.c.              6295630.2   1450695.2      6.2      1.3
MIL - Dairy products            2220653.0    822476.7      2.2      0.7
V_F - Vegetables, fruit, nuts    681876.1    999432.1      0.7      0.9
VOL - Vegetable oils and fats    539025.5    395617.7      0.5      0.4
CMT - Bovine meat prods          119645.4     23555.8      0.1      0.0
SGR - Sugar                       12164.5     35892.7      0.0      0.0
FSH - Fishing                      9106.1      1842.2      0.0      0.0
OSD - Oil seeds                    2931.7      2436.2      0.0      0.0

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 42.0% 
GTAP 34.9% 
EORA 37.6%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_27836/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,296,Kiribati,156,China,Import,Sea,151530,151530 | Vegetable oils; cast,1515,1515 | Fixed vegetable or m,15,"15 | Animal, vegetable or",2167,"Palm, coconut, palm kernel, babassu and linsee...",Agriculture,21,VOL - Vegetable oils and fats,2732.118,0,2732.118,kg,7680.0,7680.0,1368.244694,8584.0,5333.848664,7298.010135,"['Container', 'Chemical tanker']","['Container', 'Chemical tanker']",38.145452,38.078849,38.085358,38.085358,38.115204,38.115204,39.902083,70.719010,44.013276,40.862231,40.973204,41.999413,39.071147,69.258441,46.326749,42.434610,41.460819,43.338582,38.342376,75.313860,57.447392,50.241462,46.482863,45.171638,37.627106,80.077089,65.894248,61.748503,49.765886,48.412692,36.925338,81.146913,77.528695,78.195293,53.951391,50.855972,36.209503,79.479024,76.165256,83.450229,61.727559,51.788084,35.535135,70.897218,73.893773,77.432963,58.046197,52.399986,34.847494,65.125154,66.201423,82.577731,68.220793,52.746983,2167,Fats and Oils
1,1,296,Kiribati,242,Fiji,Import,Sea,151530,151530 | Vegetable oils; cast,1515,1515 | Fixed vegetable or m,15,"15 | Animal, vegetable or",2167,"Palm, coconut, palm kernel, babassu and linsee...",Agriculture,21,VOL - Vegetable oils and fats,267.018,0,267.018,kg,5.0,5.0,133.722614,3362.0,2089.049302,279.353134,"['Container', 'Chemical tanker']","['Container', 'Chemical tanker']",1.460131,1.457582,1.457831,1.457831,1.458973,1.458973,1.527371,2.706981,1.684740,1.564124,1.568372,1.607653,1.495565,2.651074,1.773295,1.624311,1.587037,1.658914,1.467669,2.882863,2.198970,1.923142,1.779270,1.729079,1.440290,3.065190,2.522299,2.363608,1.904938,1.853140,1.413428,3.106140,2.967642,2.993158,2.065151,1.946664,1.386027,3.042297,2.915453,3.194307,2.362807,1.982344,1.360213,2.713803,2.828505,2.963978,2.221892,2.005766,1.333892,2.492860,2.534057,3.160909,2.611355,2.019048,2167,Fats and Oils


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $28.5m of $101.7m (28.1%)
Volume: 32.5 of 112.9 million kilograms (28.8%)
Value-to-Weight: 0.88 from 0.9 US$ per kg (+-2.6%)
Average distance to market (volume-weighted): 4146.3 from 4207.3 nm (+-1.4%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    246  17515581.9  20270940.6     17.2     18.0
1      Sugar and Honey     41   3766777.8   7667006.7      3.7      6.8
2        Meat and Prep     14   3520101.3   1897156.9      3.5      1.7
3                Dairy     41   1875706.3    711683.8      1.8      0.6
4           Vegetables    121   1020772.9   1304566.1      1.0      1.2
5        Fats and Oils     29    537926.5    391350.3      0.5      0.3
6               Fruits     31    287266.7    252068.1      0.3      0.2
7  Unprocessed Cereals      5      3466.4     13145.9      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,1782940.8,3159524.2,1966535.0,1825805.7,1830756.1,1876619.6,1649970.3,3625361.1,3463637.4,3493497.3,2410464.2,2272248.9,1557150.7,2909943.1,2958010.6,3689163.7,3047620.2,2356738.0
1,Fats and Oils,1183732.2,2087404.3,1303186.4,1210602.4,1213889.7,1244996.2,1096393.3,2392557.8,2287101.6,2309881.3,1595575.7,1504475.7,1035428.4,1924044.0,1956685.8,2432689.0,2010025.9,1559847.1
2,Fruits,554475.8,979405.7,610736.0,567627.4,569063.9,583369.6,513405.4,1123575.6,1072126.3,1081764.7,747835.6,705655.9,484737.3,906744.8,921022.5,1142451.8,941681.8,731893.9
3,Meat and Prep,4217612.4,7474932.4,4652161.6,4319099.2,4330828.9,4439298.2,3902973.3,8577151.9,8194709.7,8265168.4,5702610.9,5375428.1,3683347.2,6883667.2,6997427.7,8728388.0,7210873.2,5575306.1
4,Processed Cereals,46384551.9,82111928.7,51138269.4,47494882.4,47620646.2,48815169.3,42932876.9,94211135.5,89975347.1,90761007.4,62661810.5,59086923.7,40523519.1,75748218.5,76980250.3,95842775.4,79122431.4,61283470.6
5,Sugar and Honey,10591703.6,17622661.4,11391708.8,10688817.1,10722380.9,11046154.7,9906607.0,20601914.2,19674972.1,20125639.6,14083949.9,13026147.3,9428392.0,16796338.7,17152793.4,20299555.2,16971432.7,13430240.3
6,Unprocessed Cereals,18348.5,32131.8,20140.8,18736.3,18788.9,19278.3,17015.1,37000.9,35350.6,35750.3,24731.3,23225.4,16084.3,29775.9,30296.3,37427.2,30992.9,24061.5
7,Vegetables,2781599.8,4841121.8,3042973.1,2840612.8,2846132.8,2919958.2,2582590.9,5559650.7,5283048.3,5344392.8,3711908.7,3514257.0,2442624.7,4557970.7,4621757.5,5607934.9,4591852.2,3643213.6


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.73,0.10,0.02,0.03,0.05,1.05,0.97,0.98,0.41,0.33,0.72,0.75,1.14,0.79,0.43
1,Fats and Oils,1.68,0.22,0.05,0.06,0.11,2.41,2.21,2.26,0.93,0.76,1.65,1.71,2.60,1.81,0.97
2,Fruits,1.48,0.20,0.05,0.05,0.10,2.12,1.94,1.98,0.82,0.67,1.47,1.52,2.29,1.59,0.86
3,Meat and Prep,0.93,0.12,0.03,0.03,0.06,1.33,1.22,1.24,0.51,0.42,0.91,0.94,1.43,1.00,0.54
4,Processed Cereals,2.04,0.27,0.06,0.07,0.14,2.93,2.69,2.73,1.13,0.92,2.01,2.08,3.16,2.20,1.19
5,Sugar and Honey,1.87,0.21,0.03,0.03,0.12,2.84,2.59,2.71,1.11,0.83,1.96,2.05,2.89,2.00,1.06
6,Unprocessed Cereals,3.98,0.52,0.11,0.13,0.27,5.77,5.29,5.40,2.23,1.79,3.95,4.10,6.16,4.30,2.30
7,Vegetables,2.02,0.26,0.06,0.06,0.14,2.92,2.65,2.71,1.11,0.91,2.07,2.13,3.10,2.11,1.18
